# TripoSR on Colab — image → 3D model in seconds
Open image-to-3D (VAST-AI-Research/TripoSR). Upload concept art or a
photo, get a GLB/OBJ back. Runs fine on a free T4.

**Phone steps:** Runtime → Run all → open the gradio link → upload image → download model.

In [ ]:
# 1) TripoSR + deps
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!git clone https://github.com/VAST-AI-Research/TripoSR
%cd TripoSR
!pip install -q -r requirements.txt
!pip install -q gradio rembg onnxruntime trimesh

In [ ]:
# 2) weights (auto-download from HF on first run; pin here explicitly)
from huggingface_hub import snapshot_download
snapshot_download('stabilityai/TripoSR', local_dir='weights')
print('weights ok')

In [ ]:
# 3) Gradio UI: image in -> GLB + OBJ out
import gradio as gr, torch, numpy as np, trimesh
from PIL import Image
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation

model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.cuda(); model.renderer.set_chunk_size(8192)

def image_to_3d(img, mc_resolution=256):
    img = remove_background(img)
    img = resize_foreground(img, 0.85)
    scene = model.reconstruct([img], chunk_size=8192)
    mesh = to_gradio_3d_orientation(scene[0].get_mesh()).as_mesh()
    mesh.export('/tmp/tripo_out.glb'); mesh.export('/tmp/tripo_out.obj')
    return '/tmp/tripo_out.glb', '/tmp/tripo_out.obj'

gr.Interface(fn=image_to_3d,
             inputs=[gr.Image(type='pil', label='Concept art / photo'),
                     gr.Slider(64, 512, value=256, step=64, label='Marching-cubes resolution')],
             outputs=[gr.Model3D(label='GLB preview'), gr.File(label='OBJ download')],
             title='TripoSR — image to 3D').launch(share=True)

## Notes
- Best input: clean character/prop on plain background (rembg runs first anyway).
- Output is untextured-ish geometry — detail it in Blender, paint in ArmorPaint.
- Heavier alternative: TRELLIS (better quality, needs more VRAM) — same notebook pattern.